# Initializing files and speakers

In [ ]:
pilar_files = {
    "pilar-i014": ["Speaker1", "2"],
    "pilar-i015": ["Speaker2", "Speaker3"],
    "pilar-i016": ["Speaker4", "1"],
    "pilar-i017": ["Speaker5"],
    "pilar-a017_FIN": ["Speaker6", "Speaker7", "Speaker2"],
    "pilar-i025": ["A", "B"],
}

# Loading dictionary and lexicon

In [ ]:
import xml.etree.ElementTree as ET
import json

In [ ]:
#loading dictionary

dict_path = "word_morph_gloss_dict.json"

with open(dict_path, "r", encoding="utf-8") as f:
    word_morph_gloss_dict = json.load(f)

print(f"Loaded dictionary with {len(word_morph_gloss_dict)} entries.")

In [ ]:
clitics_dict = {
    "l=": "CLIT",
    "j=": "CLIT",
    "m=": "CLIT",
    "n=": "CLIT",
    "s=": "IMPERS",
    "sa=": "IMPERS",
    "se=": "REFL",
    "as=": "?",
    "=la": "3SG.F.ACC",
    "=lo": "3SG.M.ACC",
    "a=": "3?",
    "i=": "1?",
}

In [ ]:
#loading the lexicon

from collections import defaultdict

def load_lexicon_dict(lexicon_path, lang_preference=None):
    """
    Load ELAN-style lexicon (Lexicon-Pilar) into a dict:

        lexicon_dict[token_form] = gloss_string

    - token_form is taken from <lexical-unit>.
    - If multiple senses/glosses exist:
        * We collect all gloss strings (optionally filtered by @lang == lang_preference).
        * If there is exactly ONE unique gloss string, we keep it.
        * If there are multiple different glosses, we treat it as ambiguous and skip that form.

    lang_preference:
      - None: use all glosses regardless of @lang
      - "spa" or "piem" etc.: only use gloss elements with that lang.
    """
    tree = ET.parse(lexicon_path)
    root = tree.getroot()

    # In your Lexicon-Pilar file, the root is likely <lexicon>, no namespaces needed.
    lexicon_raw = defaultdict(list)

    for entry in root.findall("entry"):
        lu_el = entry.find("lexical-unit")
        if lu_el is None or not (lu_el.text or "").strip():
            continue

        form = lu_el.text.strip()

        for sense in entry.findall("sense"):
            gloss_el = sense.find("gloss")
            if gloss_el is None or gloss_el.text is None:
                continue

            g_text = gloss_el.text.strip()
            g_lang = gloss_el.get("lang")  # may be "spa", "piem", or None

            if not g_text:
                continue

            # If lang_preference is specified, only accept matching glosses
            if lang_preference is not None and g_lang != lang_preference:
                continue

            lexicon_raw[form].append(g_text)

    # Resolve to unique glosses
    lexicon_dict = {}
    for form, glosses in lexicon_raw.items():
        unique_glosses = sorted(set(glosses))
        if len(unique_glosses) == 1:
            lexicon_dict[form] = unique_glosses[0]
        else:
            # ambiguous: multiple different gloss strings, we skip that form
            # (you could log them if you want)
            # print(f"Ambiguous lexicon entry for '{form}': {unique_glosses}, skipping.")
            pass

    print(f"Loaded {len(lexicon_dict)} unambiguous entries from lexicon.")
    return lexicon_dict


In [ ]:
lexicon_path = "Lexicon-Pilar"
lexicon_dict = load_lexicon_dict(lexicon_path, lang_preference=None)

# Clitics fix

In [ ]:
def split_equals(s):
    """
    Split a string containing clitics with '='.
    Rules:
      - If string ends with '=la' or '=lo', pull that off as final token.
      - Then split remaining part by '=', keeping '=' with the preceding segment:
          'a=s=parlava' -> ['a=', 's=', 'parlava']
          'cante=la'    -> ['cante', '=la']
    """
    s = s.strip()
    out = []

    # Step 1: separate final =la / =lo if present
    final_suffix = None
    if s.endswith("=la") or s.endswith("=lo"):
        final_suffix = s[-3:]  # '=la' or '=lo'
        s = s[:-3]             # remaining part

    # Step 2: split remaining on '='
    if "=" not in s:
        if s:
            out.append(s)
    else:
        parts = s.split("=")
        # 'a=s=parlava' -> ['a','s','parlava']
        for i, p in enumerate(parts):
            if i < len(parts) - 1:
                out.append(p + "=")
            else:
                if p:
                    out.append(p)

    # Step 3: append final suffix if present
    if final_suffix:
        out.append(final_suffix)

    return out

In [ ]:
def clitics(inputfilename, outputfilename, config):
    """
    Split clitic-marked tokens (with '=') into multiple annotations on the TOKEN tier.

    For each token tier in config["triplets"]:
      - Any token whose text contains '=' will be:
          * split into several new token annotations (same parent REF),
          * and its corresponding morph, gloss, and language annotations
            (for that token) will be deleted.

    This should be run BEFORE populate_missing_morph_gloss_from_tokens(),
    so that the morph+gloss tiers can be freshly rebuilt from the dictionary.

    """
    filename = inputfilename
    triplets = config["triplets"]

    print(f"\n=== Running clitics() on file: {filename} ===")

    tree = ET.parse(filename)
    root = tree.getroot()

    # ----- Namespace handling -----
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}", 1)[0][1:]
    else:
        ns_uri = None

    def q(tag):
        return f"{{{ns_uri}}}{tag}" if ns_uri else tag

    # ----- Find current max annotation id -----
    max_id_num = 0
    for ref in root.findall(f".//{q('REF_ANNOTATION')}"):
        aid = ref.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))
    for al in root.findall(f".//{q('ALIGNABLE_ANNOTATION')}"):
        aid = al.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))

    def new_ann_id():
        nonlocal max_id_num
        max_id_num += 1
        return f"a{max_id_num}"

    def find_tier(tier_name):
        for t in root.findall(f".//{q('TIER')}"):
            if t.get("TIER_ID") == tier_name:
                return t
        return None

    total_tokens = 0
    split_tokens = 0

    # ==========================================
    # Process each (token, morph, gloss, lang) group
    # ==========================================
    for (token_tiername, morph_tiername, gloss_tiername, lang_tiername) in triplets:
        print(f"\n--- Token tier: {token_tiername} ---")

        token_tier = find_tier(token_tiername)
        morph_tier = find_tier(morph_tiername)
        gloss_tier = find_tier(gloss_tiername)
        lang_tier  = find_tier(lang_tiername)

        if token_tier is None:
            print(f"  ⚠ Token tier '{token_tiername}' not found. Skipping.")
            continue

        # ---------- 1) Build maps for dependent tiers ----------
        # token_id → list of morph_ids
        token_to_morph_ids = {}
        # morph_id → list of gloss ANN elements
        gloss_by_morph = {}
        # morph_id → list of language ANN elements
        lang_by_morph = {}

        if morph_tier is not None:
            for ann in morph_tier.findall(q("ANNOTATION")):
                ref = ann.find(q("REF_ANNOTATION"))
                if ref is None:
                    continue
                mid = ref.get("ANNOTATION_ID")
                parent = ref.get("ANNOTATION_REF")
                if mid and parent:
                    token_to_morph_ids.setdefault(parent, []).append(mid)

        if gloss_tier is not None:
            for ann in gloss_tier.findall(q("ANNOTATION")):
                ref = ann.find(q("REF_ANNOTATION"))
                if ref is None:
                    continue
                mid = ref.get("ANNOTATION_REF")
                if mid:
                    gloss_by_morph.setdefault(mid, []).append(ann)

        if lang_tier is not None:
            for ann in lang_tier.findall(q("ANNOTATION")):
                ref = ann.find(q("REF_ANNOTATION"))
                if ref is None:
                    continue
                mid = ref.get("ANNOTATION_REF")
                if mid:
                    lang_by_morph.setdefault(mid, []).append(ann)

        # ---------- 2) Split tokens with '=' on token tier ----------
        original_anns = list(token_tier.findall(q("ANNOTATION")))
        new_children = []
        tokens_to_clear = set()  # token IDs whose morph/gloss/lang must be removed
        parent_to_new_ids = {}   # parent_ref -> [new_token_ids] (for recomputing PREVIOUS_ANNOTATION)

        for ann in original_anns:
            ref_el = ann.find(q("REF_ANNOTATION"))
            align_el = ann.find(q("ALIGNABLE_ANNOTATION"))

            # Dependent token tier (usual case)
            if ref_el is not None:
                val_el = ref_el.find(q("ANNOTATION_VALUE"))
                text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
                tid = ref_el.get("ANNOTATION_ID")
                parent_ref = ref_el.get("ANNOTATION_REF")

                total_tokens += 1

                if "=" not in text:
                    # Keep as-is: also track in parent_to_new_ids
                    new_children.append(ann)
                    if parent_ref and tid:
                        parent_to_new_ids.setdefault(parent_ref, []).append(tid)
                    continue

                # Need to split this token
                parts = split_equals(text)
                split_tokens += 1
                tokens_to_clear.add(tid)

                # Create fresh annotations for each part (new IDs)
                for part in parts:
                    new_ann = ET.Element(q("ANNOTATION"))
                    new_id = new_ann_id()
                    new_ref = ET.SubElement(new_ann, q("REF_ANNOTATION"), {
                        "ANNOTATION_ID": new_id,
                        "ANNOTATION_REF": parent_ref,
                    })
                    val2 = ET.SubElement(new_ref, q("ANNOTATION_VALUE"))
                    val2.text = part

                    new_children.append(new_ann)
                    if parent_ref:
                        parent_to_new_ids.setdefault(parent_ref, []).append(new_id)

            # Alignable token tier (if token tier happens to be alignable)
            elif align_el is not None:
                val_el = align_el.find(q("ANNOTATION_VALUE"))
                text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
                total_tokens += 1

                if "=" not in text:
                    new_children.append(ann)
                    continue

                parts = split_equals(text)
                split_tokens += 1

                ts1 = align_el.get("TIME_SLOT_REF1")
                ts2 = align_el.get("TIME_SLOT_REF2")

                for part in parts:
                    new_ann = ET.Element(q("ANNOTATION"))
                    new_align_id = new_ann_id()
                    new_align = ET.SubElement(new_ann, q("ALIGNABLE_ANNOTATION"), {
                        "ANNOTATION_ID": new_align_id,
                        "TIME_SLOT_REF1": ts1,
                        "TIME_SLOT_REF2": ts2,
                    })
                    val2 = ET.SubElement(new_align, q("ANNOTATION_VALUE"))
                    val2.text = part
                    new_children.append(new_ann)

            else:
                # Unexpected structure: keep unchanged
                new_children.append(ann)

        # ---------- 3) Recompute PREVIOUS_ANNOTATION chains ----------
        prev_map = {}  # annotation_id -> previous_id

        for parent_ref, id_list in parent_to_new_ids.items():
            prev = None
            for aid in id_list:
                prev_map[aid] = prev
                prev = aid

        for ann in new_children:
            ref_el = ann.find(q("REF_ANNOTATION"))
            if ref_el is None:
                continue
            aid = ref_el.get("ANNOTATION_ID")
            if aid in prev_map:
                prev_id = prev_map[aid]
                # remove old PREVIOUS_ANNOTATION if existed
                if "PREVIOUS_ANNOTATION" in ref_el.attrib:
                    del ref_el.attrib["PREVIOUS_ANNOTATION"]
                if prev_id is not None:
                    ref_el.set("PREVIOUS_ANNOTATION", prev_id)

        # Replace children on token tier
        for ann in original_anns:
            token_tier.remove(ann)
        for ann in new_children:
            token_tier.append(ann)

        print(f"  Processed {len(original_anns)} annotations on '{token_tiername}'.")
        print(f"  Tokens with '=' split in this tier: {len(tokens_to_clear)}")

        # ---------- 4) Clear morph/gloss/lang for affected tokens ----------
        if tokens_to_clear and morph_tier is not None:
            removed_morph_ids = set()

            # Remove morph annotations whose parent is in tokens_to_clear
            for ann in list(morph_tier.findall(q("ANNOTATION"))):
                ref = ann.find(q("REF_ANNOTATION"))
                if ref is None:
                    continue
                parent = ref.get("ANNOTATION_REF")
                mid = ref.get("ANNOTATION_ID")
                if parent in tokens_to_clear:
                    morph_tier.remove(ann)
                    if mid:
                        removed_morph_ids.add(mid)

            # Remove gloss annotations whose REF points to those morphs
            if gloss_tier is not None:
                for ann in list(gloss_tier.findall(q("ANNOTATION"))):
                    ref = ann.find(q("REF_ANNOTATION"))
                    if ref is None:
                        continue
                    mid = ref.get("ANNOTATION_REF")
                    if mid in removed_morph_ids:
                        gloss_tier.remove(ann)

            # Remove language annotations whose REF points to those morphs
            if lang_tier is not None:
                for ann in list(lang_tier.findall(q("ANNOTATION"))):
                    ref = ann.find(q("REF_ANNOTATION"))
                    if ref is None:
                        continue
                    mid = ref.get("ANNOTATION_REF")
                    if mid in removed_morph_ids:
                        lang_tier.remove(ann)

            print(f"  Cleared morph/gloss/lang for {len(tokens_to_clear)} tokens in this triplet.")

    # ---------- 5) Update lastUsedAnnotationId ----------
    header = root.find(q("HEADER"))
    if header is not None:
        prop_last = None
        for prop in header.findall(q("PROPERTY")):
            if prop.get("NAME") == "lastUsedAnnotationId":
                prop_last = prop
                break
        if prop_last is None:
            prop_last = ET.SubElement(header, q("PROPERTY"), {"NAME": "lastUsedAnnotationId"})
        prop_last.text = str(max_id_num)

    # ---------- 6) Save ----------
    tree.write(outputfilename, encoding="utf-8", xml_declaration=True)

    print("\n=== clitics() SUMMARY ===")
    print(f"Total token annotations seen (these tiers): {total_tokens}")
    print(f"Tokens that contained '=' and were split: {split_tokens}")
    print(f"New file saved as: {outputfilename}")

In [ ]:
def clitics_reverse(inputfilename, outputfilename, config, clitics_dict):
    """
    To be run AFTER clitics().

    Parameters
    ----------
    inputfilename : str
        Path to the input .eaf file (already processed by clitics()).
    outputfilename : str
        Path to save the modified .eaf file.
    config : dict
        Only the "triplets" key is used:
        {
            "triplets": [
                (token_tier, morph_tier, gloss_tier, language_tier),
                ...
            ]
        }
    clitics_dict : dict
        Mapping from clitic form (e.g. "a=", "=la") to gloss (e.g. "3SG", "CL.F.SG").

    Behavior
    --------
    For each triplet (token, morph, gloss, lang):

      For every token whose text contains '=':
        (1) Copy the token value onto the morph tier (one morph annotation).
        (2) Fill in the gloss tier according to clitics_dict (gloss "" if not found).
        (3) Remove '=' from the token tier text (e.g. "a=" -> "a", "=la" -> "la").
    """
    triplets = config["triplets"]

    print(f"\n=== Running clitics_reverse() on file: {inputfilename} ===")

    tree = ET.parse(inputfilename)
    root = tree.getroot()

    # ----- Namespace handling -----
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}", 1)[0][1:]
    else:
        ns_uri = None

    def q(tag):
        return f"{{{ns_uri}}}{tag}" if ns_uri else tag

    # ----- Find current max annotation id -----
    max_id_num = 0
    for ref in root.findall(f".//{q('REF_ANNOTATION')}"):
        aid = ref.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))
    for al in root.findall(f".//{q('ALIGNABLE_ANNOTATION')}"):
        aid = al.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))

    def new_ann_id():
        nonlocal max_id_num
        max_id_num += 1
        return f"a{max_id_num}"

    def find_tier(tier_name):
        for t in root.findall(f".//{q('TIER')}"):
            if t.get("TIER_ID") == tier_name:
                return t
        return None

    total_clitics = 0
    glossed_from_dict = 0
    gloss_missing = 0

    # ==========================================
    # Process each (token, morph, gloss, lang) group
    # ==========================================
    for (token_tiername, morph_tiername, gloss_tiername, _lang_tiername) in triplets:
        print(f"\n--- Triplet: {token_tiername} / {morph_tiername} / {gloss_tiername} ---")

        token_tier = find_tier(token_tiername)
        morph_tier = find_tier(morph_tiername)
        gloss_tier = find_tier(gloss_tiername)

        if token_tier is None:
            print(f"  ⚠ Token tier '{token_tiername}' not found. Skipping.")
            continue
        if morph_tier is None or gloss_tier is None:
            print(f"  ⚠ Morph or gloss tier not found. Skipping this triplet.")
            continue

        # Iterate over token annotations
        for ann in token_tier.findall(q("ANNOTATION")):
            ref_el = ann.find(q("REF_ANNOTATION"))
            align_el = ann.find(q("ALIGNABLE_ANNOTATION"))

            # We assume token tiers are dependent (REF_ANNOTATION)
            if ref_el is None and align_el is not None:
                # If you ever have alignable tokens, we could handle them separately
                continue
            if ref_el is None:
                continue

            val_el = ref_el.find(q("ANNOTATION_VALUE"))
            text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
            token_id = ref_el.get("ANNOTATION_ID")

            if "=" not in text or not token_id:
                continue  # not a clitic token

            total_clitics += 1

            # (1) Create morph annotation: copy token text (with '=')
            morph_ann = ET.SubElement(morph_tier, q("ANNOTATION"))
            morph_id = new_ann_id()
            morph_ref = ET.SubElement(morph_ann, q("REF_ANNOTATION"), {
                "ANNOTATION_ID": morph_id,
                "ANNOTATION_REF": token_id,
            })
            morph_val = ET.SubElement(morph_ref, q("ANNOTATION_VALUE"))
            morph_val.text = text

            # (2) Create gloss annotation using clitics_dict
            gloss_ann = ET.SubElement(gloss_tier, q("ANNOTATION"))
            gloss_id = new_ann_id()
            gloss_ref = ET.SubElement(gloss_ann, q("REF_ANNOTATION"), {
                "ANNOTATION_ID": gloss_id,
                "ANNOTATION_REF": morph_id,
            })
            gloss_val = ET.SubElement(gloss_ref, q("ANNOTATION_VALUE"))

            gloss_text = clitics_dict.get(text, "")
            if gloss_text:
                glossed_from_dict += 1
            else:
                gloss_missing += 1
            gloss_val.text = gloss_text

            # (3) Remove '=' from token tier text (e.g. "a=" -> "a", "=la" -> "la")
            clean_text = text.replace("=", "")
            val_el.text = clean_text

        print(f"  Clitic tokens processed on this triplet so far: {total_clitics}")

    # ---------- Update lastUsedAnnotationId ----------
    header = root.find(q("HEADER"))
    if header is not None:
        prop_last = None
        for prop in header.findall(q("PROPERTY")):
            if prop.get("NAME") == "lastUsedAnnotationId":
                prop_last = prop
                break
        if prop_last is None:
            prop_last = ET.SubElement(header, q("PROPERTY"), {"NAME": "lastUsedAnnotationId"})
        prop_last.text = str(max_id_num)

    # ---------- Save ----------
    tree.write(outputfilename, encoding="utf-8", xml_declaration=True)

    print("\n=== clitics_reverse() SUMMARY ===")
    print(f"Total clitic tokens found (with '='): {total_clitics}")
    print(f"  Glossed via clitics_dict: {glossed_from_dict}")
    print(f"  Clitics without dictionary gloss (gloss=''): {gloss_missing}")
    print(f"New file saved as: {outputfilename}")


# Glossing

In [ ]:
import xml.etree.ElementTree as ET
import json

In [ ]:
def populate_missing_morph_gloss_from_tokens(
    inputfilename,
    outputfilename,
    config,
    word_morph_gloss_dict
):
    """
    Fill morph and gloss tiers from the token tier, BUT:

      • If a token already has ANY *non-empty* gloss annotation on its morphs,
        that token is completely skipped (no changes to its morph/gloss/lang).
      • Tokens with NO gloss, or only EMPTY glosses, are (re)populated.

    For tokens that are repopulated:
      • Old morph, gloss, and language annotations for that token are removed.
      • New morph+gloss sequences are created from word_morph_gloss_dict.
      • If word not in dict -> one morph = token text, gloss = "".

    Dictionary format:
      word_morph_gloss_dict[word] = [[morph1, gloss1], [morph2, gloss2], ...]

    Config format:
      config = {
          "triplets": [
              (token_tier, morph_tier, gloss_tier, language_tier),
              ...
          ]
      }
    """
    triplets = config["triplets"]

    print(f"\n=== populate_missing_morph_gloss_from_tokens() on {inputfilename} ===")

    tree = ET.parse(inputfilename)
    root = tree.getroot()

    # ----- Namespace handling -----
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}", 1)[0][1:]
    else:
        ns_uri = None

    def q(tag):
        return f"{{{ns_uri}}}{tag}" if ns_uri else tag

    # ----- Find current max annotation id -----
    max_id_num = 0
    for ref in root.findall(f".//{q('REF_ANNOTATION')}"):
        aid = ref.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))
    for al in root.findall(f".//{q('ALIGNABLE_ANNOTATION')}"):
        aid = al.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))

    def new_ann_id():
        nonlocal max_id_num
        max_id_num += 1
        return f"a{max_id_num}"

    def find_tier(name):
        for t in root.findall(f".//{q('TIER')}"):
            if t.get("TIER_ID") == name:
                return t
        return None

    total_tokens = 0
    skipped_tokens_with_gloss = 0
    populated_tokens = 0

    # ==========================================
    # Process each (token, morph, gloss, lang) group
    # ==========================================
    for (token_tiername, morph_tiername, gloss_tiername, lang_tiername) in triplets:
        print(f"\n--- Triplet: {token_tiername} / {morph_tiername} / {gloss_tiername} / {lang_tiername} ---")

        token_tier = find_tier(token_tiername)
        morph_tier = find_tier(morph_tiername)
        gloss_tier = find_tier(gloss_tiername)
        lang_tier  = find_tier(lang_tiername)

        if token_tier is None or morph_tier is None or gloss_tier is None:
            print("  ⚠ Missing token/morph/gloss tier, skipping this triplet.")
            continue

        # ---------- Build maps ----------
        # token_id -> [morph_ann elements]
        morphs_by_token = {}
        for ann in morph_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                continue
            mid = ref.get("ANNOTATION_ID")
            parent = ref.get("ANNOTATION_REF")
            if not mid or not parent:
                continue
            morphs_by_token.setdefault(parent, []).append(ann)

        # morph_id -> list of (gloss_ann_element, gloss_text)
        glosses_by_morph = {}
        if gloss_tier is not None:
            for ann in gloss_tier.findall(q("ANNOTATION")):
                ref = ann.find(q("REF_ANNOTATION"))
                if ref is None:
                    continue
                mid = ref.get("ANNOTATION_REF")
                if not mid:
                    continue
                val = ref.find(q("ANNOTATION_VALUE"))
                txt = (val.text or "") if (val is not None and val.text is not None) else ""
                glosses_by_morph.setdefault(mid, []).append((ann, txt.strip()))

        # morph_id -> [language_ann elements]
        langs_by_morph = {}
        if lang_tier is not None:
            for ann in lang_tier.findall(q("ANNOTATION")):
                ref = ann.find(q("REF_ANNOTATION"))
                if ref is None:
                    continue
                mid = ref.get("ANNOTATION_REF")
                if not mid:
                    continue
                langs_by_morph.setdefault(mid, []).append(ann)

        # ---------- Iterate over token annotations ----------
        for ann in token_tier.findall(q("ANNOTATION")):
            ref_el = ann.find(q("REF_ANNOTATION"))

            # We assume token tiers are dependent (REF_ANNOTATION)
            if ref_el is None:
                continue

            total_tokens += 1

            val_el = ref_el.find(q("ANNOTATION_VALUE"))
            token_text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
            token_id = ref_el.get("ANNOTATION_ID")
            if not token_text or not token_id:
                continue

            # ----- NEW RULE: skip if token already has ANY *non-empty* gloss -----
            morph_anns = morphs_by_token.get(token_id, [])
            has_nonempty_gloss = False
            for morph_ann in morph_anns:
                mref = morph_ann.find(q("REF_ANNOTATION"))
                mid = mref.get("ANNOTATION_ID") if mref is not None else None
                if not mid:
                    continue
                for _gann, gtxt in glosses_by_morph.get(mid, []):
                    if gtxt.strip():  # at least one non-empty gloss
                        has_nonempty_gloss = True
                        break
                if has_nonempty_gloss:
                    break

            if has_nonempty_gloss:
                skipped_tokens_with_gloss += 1
                continue  # do not touch these tokens at all

            # At this point, token has either no gloss annotations at all,
            # or only empty gloss values. We *do* want to repopulate it.

            # ----- Remove old gloss/lang/morph for this token -----
            morph_ids = []
            for morph_ann in morph_anns:
                mref = morph_ann.find(q("REF_ANNOTATION"))
                mid = mref.get("ANNOTATION_ID") if mref is not None else None
                if mid:
                    morph_ids.append(mid)
            morph_ids_set = set(morph_ids)

            # Remove gloss annotations for these morphs
            if gloss_tier is not None and morph_ids_set:
                for gloss_ann in list(gloss_tier.findall(q("ANNOTATION"))):
                    ref = gloss_ann.find(q("REF_ANNOTATION"))
                    if ref is None:
                        continue
                    mid = ref.get("ANNOTATION_REF")
                    if mid in morph_ids_set:
                        gloss_tier.remove(gloss_ann)

            # Remove language annotations for these morphs
            if lang_tier is not None and morph_ids_set:
                for lang_ann in list(lang_tier.findall(q("ANNOTATION"))):
                    ref = lang_ann.find(q("REF_ANNOTATION"))
                    if ref is None:
                        continue
                    mid = ref.get("ANNOTATION_REF")
                    if mid in morph_ids_set:
                        lang_tier.remove(lang_ann)

            # Remove morph annotations themselves
            for morph_ann in morph_anns:
                morph_tier.remove(morph_ann)

            # ----- Decide sequence from dictionary -----
            pair_seq = word_morph_gloss_dict.get(token_text)
            if pair_seq is None:
                # token not in dict → one morph = token text, gloss empty
                pair_seq = [[token_text, ""]]

            # ----- Create new morph + gloss annotations -----
            prev_morph_id = None
            for morph_text, gloss_text in pair_seq:
                # morph
                morph_ann = ET.SubElement(morph_tier, q("ANNOTATION"))
                morph_id = new_ann_id()
                attrs = {"ANNOTATION_ID": morph_id, "ANNOTATION_REF": token_id}
                if prev_morph_id is not None:
                    attrs["PREVIOUS_ANNOTATION"] = prev_morph_id
                mref = ET.SubElement(morph_ann, q("REF_ANNOTATION"), attrs)
                mval = ET.SubElement(mref, q("ANNOTATION_VALUE"))
                mval.text = morph_text

                # gloss
                gloss_ann = ET.SubElement(gloss_tier, q("ANNOTATION"))
                gloss_id = new_ann_id()
                gref = ET.SubElement(gloss_ann, q("REF_ANNOTATION"), {
                    "ANNOTATION_ID": gloss_id,
                    "ANNOTATION_REF": morph_id
                })
                gval = ET.SubElement(gref, q("ANNOTATION_VALUE"))
                gval.text = gloss_text or ""

                prev_morph_id = morph_id

            populated_tokens += 1

        print(f"  Tokens in this triplet: {total_tokens}")
        print(f"  Skipped (had non-empty gloss): {skipped_tokens_with_gloss}")
        print(f"  Populated so far: {populated_tokens}")

    # ---------- Update lastUsedAnnotationId ----------
    header = root.find(q("HEADER"))
    if header is not None:
        prop_last = None
        for prop in header.findall(q("PROPERTY")):
            if prop.get("NAME") == "lastUsedAnnotationId":
                prop_last = prop
                break
        if prop_last is None:
            prop_last = ET.SubElement(header, q("PROPERTY"), {"NAME": "lastUsedAnnotationId"})
        prop_last.text = str(max_id_num)

    # ---------- Save ----------
    tree.write(outputfilename, encoding="utf-8", xml_declaration=True)

    print("\n=== SUMMARY populate_missing_morph_gloss_from_tokens ===")
    print(f"Total tokens seen (all triplets): {total_tokens}")
    print(f"Tokens skipped (had non-empty gloss): {skipped_tokens_with_gloss}")
    print(f"Tokens populated (dict or fallback): {populated_tokens}")
    print(f"New file saved as: {outputfilename}")


# Fixing clitics glosses

In [ ]:
def fix_person_number_glosses_by_token(inputfilename, outputfilename, config):
    """
    For each (token, morph, gloss, lang) triplet in config["triplets"]:

    If a token has a gloss '1?', '2?' or '3?' (on any of its morphs):

      - Look at the NEXT and the ONE AFTER NEXT token in the SAME token tier.
      - Collect all glosses of their morphs.
      - If any of those glosses ends with 'SG' or 'PL' (e.g. '3SG', 'be:3SG', 'OBJ.PL'):
            replace '?' in the current token's gloss with that number:
                '1?' -> '1SG' or '1PL'
                '2?' -> '2SG' or '2PL'
                '3?' -> '3SG' or '3PL'
      - If no such gloss is found, leave '1?/2?/3?' unchanged.

    Only modifies gloss text, does not change structure/IDs.
    """

    triplets = config["triplets"]

    tree = ET.parse(inputfilename)
    root = tree.getroot()

    # ----- Namespace handling -----
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}", 1)[0][1:]
    else:
        ns_uri = None

    def q(tag):
        return f"{{{ns_uri}}}{tag}" if ns_uri else tag

    def find_tier(name):
        for t in root.findall(f".//{q('TIER')}"):
            if t.get("TIER_ID") == name:
                return t
        return None

    targets = {"1?", "2?", "3?"}

    for (token_tiername, morph_tiername, gloss_tiername, _lang_tiername) in triplets:
        token_tier = find_tier(token_tiername)
        morph_tier = find_tier(morph_tiername)
        gloss_tier = find_tier(gloss_tiername)

        if token_tier is None or morph_tier is None or gloss_tier is None:
            print(f"⚠ Missing tier in triplet {token_tiername}/{morph_tiername}/{gloss_tiername}, skipping.")
            continue

        print(f"\nProcessing triplet: {token_tiername} / {morph_tiername} / {gloss_tiername}")

        # ---------------------------
        # 1. Build mapping: token_id -> [morph_ids]
        # ---------------------------
        morphs_by_token = {}
        for ann in morph_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                continue
            mid = ref.get("ANNOTATION_ID")
            parent_token_id = ref.get("ANNOTATION_REF")
            if not mid or not parent_token_id:
                continue
            morphs_by_token.setdefault(parent_token_id, []).append(mid)

        # ---------------------------
        # 2. Build mapping: morph_id -> list of (val_element, text)
        # ---------------------------
        glosses_by_morph = {}
        for ann in gloss_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                continue
            morph_id = ref.get("ANNOTATION_REF")
            if not morph_id:
                continue
            val_el = ref.find(q("ANNOTATION_VALUE"))
            txt = (val_el.text or "") if (val_el is not None and val_el.text is not None) else ""
            glosses_by_morph.setdefault(morph_id, []).append((val_el, txt.strip()))

        # ---------------------------
        # 3. Collect tokens in order: each with its token_id and gloss entries
        # ---------------------------
        tokens = []  # list of dicts: {"token_id": ..., "gloss_entries": [(val_el, txt), ...]}
        for ann in token_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                continue  # we assume dependent token tier
            token_id = ref.get("ANNOTATION_ID")
            if not token_id:
                continue

            # Collect gloss entries via morphs
            gloss_entries = []
            for mid in morphs_by_token.get(token_id, []):
                for val_el, txt in glosses_by_morph.get(mid, []):
                    gloss_entries.append((val_el, txt))
            tokens.append({"token_id": token_id, "gloss_entries": gloss_entries})

        # ---------------------------
        # 4. For each token with 1?/2?/3? try to infer SG/PL from next tokens
        # ---------------------------
        changed = 0

        for i, tok in enumerate(tokens):
            gloss_entries = tok["gloss_entries"]
            # Which entries in this token are 1?/2?/3?
            idxs_to_fix = [idx for idx, (_val_el, txt) in enumerate(gloss_entries) if txt in targets]
            if not idxs_to_fix:
                continue

            # Look at next one and next+1 tokens
            number_tag = None  # "SG" or "PL"
            for offset in (1, 2):
                j = i + offset
                if j >= len(tokens):
                    break
                for _val2, txt2 in tokens[j]["gloss_entries"]:
                    if not txt2:
                        continue
                    if txt2.endswith("SG"):
                        number_tag = "SG"
                        break
                    if txt2.endswith("PL"):
                        number_tag = "PL"
                        break
                if number_tag is not None:
                    break

            # If we didn't find SG/PL, leave 1?/2?/3? unchanged
            if number_tag is None:
                continue

            # Otherwise update all relevant glosses in this token
            for idx in idxs_to_fix:
                val_el, txt = gloss_entries[idx]
                if val_el is None:
                    continue
                person = txt[0]  # '1', '2', or '3'
                new_txt = person + number_tag
                val_el.text = new_txt
                # update cached text as well (for consistency if reused)
                gloss_entries[idx] = (val_el, new_txt)
                changed += 1

        print(f"  Updated {changed} gloss annotations in '{gloss_tiername}' based on following tokens.")

    # ---------------------------
    # 5. Save result
    # ---------------------------
    tree.write(outputfilename, encoding="utf-8", xml_declaration=True)
    print(f"\n✅ Person/number '?' fixes saved to: {outputfilename}")



# Annotation from the Lexicon file



In [ ]:
def fill_empty_glosses_from_lexicon(inputfilename, outputfilename, config, lexicon_dict):
    """
    After all previous steps, fill ONLY EMPTY gloss annotations
    using a lexicon dictionary, taking the key from the TOKEN tier.

    For each (token, morph, gloss, lang) triplet in config["triplets"]:

      - For each token annotation:
          * Read token_text from token tier.
          * If token_text not in lexicon_dict: skip.
          * Find all morphs dependent on this token.
          * For each morph, find all gloss annotations attached to it.
          * For each gloss ANNOTATION_VALUE:
              - if text is empty/whitespace -> fill with lexicon_dict[token_text]
              - if non-empty -> leave unchanged.

    No new annotations are created; no IDs changed or removed.
    """

    tree = ET.parse(inputfilename)
    root = tree.getroot()

    # ----- Namespace handling -----
    if root.tag.startswith("{"):
        ns_uri = root.tag[1:].split("}", 1)[0]
    else:
        ns_uri = None

    def q(tag):
        return f"{{{ns_uri}}}{tag}" if ns_uri else tag

    def find_tier(name):
        for t in root.findall(f".//{q('TIER')}"):
            if t.get("TIER_ID") == name:
                return t
        return None

    triplets = config["triplets"]

    total_candidates = 0
    filled = 0

    for (token_tiername, morph_tiername, gloss_tiername, _lang_tiername) in triplets:
        token_tier = find_tier(token_tiername)
        morph_tier = find_tier(morph_tiername)
        gloss_tier = find_tier(gloss_tiername)

        if token_tier is None or morph_tier is None or gloss_tier is None:
            print(f"⚠ Missing tier in triplet {token_tiername}/{morph_tiername}/{gloss_tiername}, skipping.")
            continue

        print(f"\nProcessing triplet: {token_tiername} / {morph_tiername} / {gloss_tiername}")

        # ---------------------------
        # 1. Build mapping: token_id -> [morph_ids]
        # ---------------------------
        morphs_by_token = {}
        for ann in morph_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                continue
            mid = ref.get("ANNOTATION_ID")
            parent_token_id = ref.get("ANNOTATION_REF")
            if not mid or not parent_token_id:
                continue
            morphs_by_token.setdefault(parent_token_id, []).append(mid)

        # ---------------------------
        # 2. Build mapping: morph_id -> list of (value_element, text)
        # ---------------------------
        glosses_by_morph = {}
        for ann in gloss_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                continue
            morph_id = ref.get("ANNOTATION_REF")
            if not morph_id:
                continue
            val_el = ref.find(q("ANNOTATION_VALUE"))
            txt = (val_el.text or "") if (val_el is not None and val_el.text is not None) else ""
            glosses_by_morph.setdefault(morph_id, []).append((val_el, txt.strip()))

        # ---------------------------
        # 3. Iterate over token annotations in order
        # ---------------------------
        for ann in token_tier.findall(q("ANNOTATION")):
            ref = ann.find(q("REF_ANNOTATION"))
            if ref is None:
                # assuming your token tiers are dependent; skip alignables here
                continue

            token_id = ref.get("ANNOTATION_ID")
            if not token_id:
                continue

            val_el = ref.find(q("ANNOTATION_VALUE"))
            token_text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
            if not token_text:
                continue

            # Use token text as key into lexicon
            gloss_from_lexicon = lexicon_dict.get(token_text)
            if gloss_from_lexicon is None:
                continue  # nothing for this token in the lexicon

            # Get morph_ids for this token
            morph_ids = morphs_by_token.get(token_id, [])
            if not morph_ids:
                continue

            for mid in morph_ids:
                for val_el_g, current_txt in glosses_by_morph.get(mid, []):
                    # Only fill if currently empty
                    if current_txt.strip():
                        continue
                    if val_el_g is None:
                        continue
                    total_candidates += 1
                    val_el_g.text = gloss_from_lexicon
                    filled += 1

        print(f"  Filled {filled} glosses so far (triplet cumulative).")

    tree.write(outputfilename, encoding="utf-8", xml_declaration=True)
    print("\n=== SUMMARY fill_empty_glosses_from_lexicon ===")
    print(f"Total empty gloss slots considered: {total_candidates}")
    print(f"Total glosses filled from lexicon:  {filled}")
    print(f"Output saved to: {outputfilename}")


# Adding the language tags

In [ ]:
def annotate_language_from_morphs_config(
    inputfilename,
    outputfilename,
    config
):
    """
    For each (token, morph, gloss, language) triplet in config["triplets"]:

      For each morph annotation:

        1) If the corresponding gloss annotation value starts with an
           uppercase letter followed by a lowercase letter (Unicode-aware,
           e.g. "Turin", "Argentina"):
                 → set language = 'piem-spa'
                 (create or overwrite any existing value for this morph).

        2) Else, if the morph annotation value is a sequence of only 'x' or 'X'
           (x, xx, XXX, etc.):
                 → remove any existing language annotation for this morph
                    and do NOT create a new one.

        3) Else:
             - If there is an existing NON-EMPTY language annotation
               for this morph, preserve it as is.
             - Otherwise, classify from the morph text, considering only
               alphabetic characters (letters); punctuation etc. is ignored:
                 • If all letters are lowercase → 'piem'
                 • If all letters are uppercase → 'spa'
                 • If there are no letters, no annotation is created.

      - Does not touch any other tiers (token/morph/gloss).
      - Updates lastUsedAnnotationId.

    config format:
      config = {
          "triplets": [
              (token_tiername, morph_tiername, gloss_tiername, language_tiername),
              ...
          ]
      }
    """
    triplets = config["triplets"]

    tree = ET.parse(inputfilename)
    root = tree.getroot()

    # ----- Namespace handling -----
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}", 1)[0][1:]
    else:
        ns_uri = None

    def q(tag):
        return f"{{{ns_uri}}}{tag}" if ns_uri else tag

    def find_tier(name):
        for t in root.findall(f".//{q('TIER')}"):
            if t.get("TIER_ID") == name:
                return t
        return None

    # ----- Find current max annotation ID -----
    max_id_num = 0
    for ref in root.findall(f".//{q('REF_ANNOTATION')}"):
        aid = ref.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))
    for al in root.findall(f".//{q('ALIGNABLE_ANNOTATION')}"):
        aid = al.get("ANNOTATION_ID")
        if aid and aid.startswith("a") and aid[1:].isdigit():
            max_id_num = max(max_id_num, int(aid[1:]))

    def new_ann_id():
        nonlocal max_id_num
        max_id_num += 1
        return f"a{max_id_num}"

    # ----- Helpers -----

    def is_x_combo(morph_text):
        """All chars in the string are x or X (at least one)."""
        t = (morph_text or "").strip()
        return bool(t) and all(ch in ("x", "X") for ch in t)

    def gloss_is_proper_name(gloss_text):
        """
        Gloss starts with uppercase followed by lowercase (Unicode-aware).
        Example: 'Turin', 'Argentina'.
        """
        t = (gloss_text or "").strip()
        if len(t) < 2:
            return False
        return t[0].isupper() and t[1].islower()

    def classify_from_morph(morph_text):
        """
        Use only alphabetic characters (letters). Punctuation etc. is ignored.
        - All letters lowercase  → 'piem'
        - All letters uppercase  → 'spa'
        - If no letters or mixed case → return None (no annotation).
        """
        letters = [ch for ch in (morph_text or "") if ch.isalpha()]
        if not letters:
            return None

        all_lower = all(ch.islower() for ch in letters)
        all_upper = all(ch.isupper() for ch in letters)

        if all_lower:
            return "piem"
        if all_upper:
            return "spa"

        # mixed or something else -> no automatic assignment
        return None

    # ==========================================
    # Process each (token, morph, gloss, lang) group
    # ==========================================
    for (token_tiername, morph_tiername, gloss_tiername, lang_tiername) in triplets:
        morph_tier = find_tier(morph_tiername)
        gloss_tier = find_tier(gloss_tiername)
        lang_tier  = find_tier(lang_tiername)

        if morph_tier is None:
            print(f"⚠ Morph tier '{morph_tiername}' not found, skipping this triplet.")
            continue

        # Create language tier as child of morph tier if missing
        if lang_tier is None:
            lang_tier = ET.SubElement(root, q("TIER"), {
                "TIER_ID": lang_tiername,
                "PARENT_REF": morph_tiername
            })

        # ---- Index existing language annotations by morph REF ----
        # lang_by_morph: morph_id -> (lang_annot_element, text_value)
        lang_by_morph = {}
        for ann in lang_tier.findall(q("ANNOTATION")):
            ref_el = ann.find(q("REF_ANNOTATION"))
            if ref_el is None:
                continue
            morph_ref = ref_el.get("ANNOTATION_REF")
            val_el = ref_el.find(q("ANNOTATION_VALUE"))
            val_text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
            if morph_ref:
                lang_by_morph[morph_ref] = (ann, val_text)

        # ---- Index gloss texts by morph annotation ID (assuming gloss depends on morph) ----
        gloss_by_morph = {}
        if gloss_tier is not None:
            for ann in gloss_tier.findall(q("ANNOTATION")):
                ref_el = ann.find(q("REF_ANNOTATION"))
                if ref_el is None:
                    continue
                morph_parent_id = ref_el.get("ANNOTATION_REF")
                val_el = ref_el.find(q("ANNOTATION_VALUE"))
                gloss_text = (val_el.text or "").strip() if (val_el is not None and val_el.text) else ""
                if morph_parent_id:
                    gloss_by_morph[morph_parent_id] = gloss_text

        created = 0

        # ---- Iterate over morph annotations ----
        for ann in morph_tier.findall(q("ANNOTATION")):
            ref_ann = ann.find(q("REF_ANNOTATION"))
            if ref_ann is None:
                continue

            morph_id = ref_ann.get("ANNOTATION_ID")
            if not morph_id:
                continue

            # Morph text
            val_el = ref_ann.find(q("ANNOTATION_VALUE"))
            morph_text = (val_el.text or "") if (val_el is not None and val_el.text) else ""

            # Gloss text (if any) corresponding to this morph
            gloss_text = gloss_by_morph.get(morph_id, "")

            # Existing language annotation for this morph (if any)
            existing_lang_ann, existing_lang_text = lang_by_morph.get(
                morph_id, (None, "")
            )

            # ---------- STEP 1: proper-name gloss -> piem-spa ----------
            if gloss_is_proper_name(gloss_text):
                new_lang_value = "piem-spa"

                if existing_lang_ann is not None:
                    # Overwrite existing value
                    ref_el_lang = existing_lang_ann.find(q("REF_ANNOTATION"))
                    if ref_el_lang is None:
                        # If structure is weird, recreate cleanly
                        lang_tier.remove(existing_lang_ann)
                        existing_lang_ann = None
                    else:
                        val_el_lang = ref_el_lang.find(q("ANNOTATION_VALUE"))
                        if val_el_lang is None:
                            val_el_lang = ET.SubElement(ref_el_lang, q("ANNOTATION_VALUE"))
                        val_el_lang.text = new_lang_value
                        lang_by_morph[morph_id] = (existing_lang_ann, new_lang_value)
                        continue  # done for this morph

                if existing_lang_ann is None:
                    # Create new language annotation
                    lang_ann = ET.SubElement(lang_tier, q("ANNOTATION"))
                    lang_id = new_ann_id()
                    lang_ref = ET.SubElement(lang_ann, q("REF_ANNOTATION"), {
                        "ANNOTATION_ID": lang_id,
                        "ANNOTATION_REF": morph_id
                    })
                    lang_val = ET.SubElement(lang_ref, q("ANNOTATION_VALUE"))
                    lang_val.text = new_lang_value
                    lang_by_morph[morph_id] = (lang_ann, new_lang_value)
                    created += 1

                continue  # next morph

            # ---------- STEP 2: morph is x/X combo -> remove lang ----------
            if is_x_combo(morph_text):
                if existing_lang_ann is not None:
                    lang_tier.remove(existing_lang_ann)
                    lang_by_morph.pop(morph_id, None)
                # No new annotation created
                continue  # next morph

            # ---------- STEP 3: preserve existing non-empty, else classify ----------
            if existing_lang_ann is not None and existing_lang_text:
                # Non-empty existing annotation -> keep as is
                continue

            # Need to assign a language from morph text
            lang_value = classify_from_morph(morph_text)
            if not lang_value:
                # nothing assigned (e.g. mixed case or no letters)
                continue

            if existing_lang_ann is not None:
                # Update empty existing annotation
                ref_el_lang = existing_lang_ann.find(q("REF_ANNOTATION"))
                if ref_el_lang is None:
                    # If broken, recreate clean
                    lang_tier.remove(existing_lang_ann)
                    existing_lang_ann = None
                else:
                    val_el_lang = ref_el_lang.find(q("ANNOTATION_VALUE"))
                    if val_el_lang is None:
                        val_el_lang = ET.SubElement(ref_el_lang, q("ANNOTATION_VALUE"))
                    val_el_lang.text = lang_value
                    lang_by_morph[morph_id] = (existing_lang_ann, lang_value)
                    continue  # updated, go to next morph

            if existing_lang_ann is None:
                # Create new language annotation
                lang_ann = ET.SubElement(lang_tier, q("ANNOTATION"))
                lang_id = new_ann_id()
                lang_ref = ET.SubElement(lang_ann, q("REF_ANNOTATION"), {
                    "ANNOTATION_ID": lang_id,
                    "ANNOTATION_REF": morph_id
                })
                lang_val = ET.SubElement(lang_ref, q("ANNOTATION_VALUE"))
                lang_val.text = lang_value
                lang_by_morph[morph_id] = (lang_ann, lang_value)
                created += 1

        print(f"Triplet '{morph_tiername} / {lang_tiername}': created {created} new language annotations.")

    # ---------- Update lastUsedAnnotationId ----------
    header = root.find(q("HEADER")) if ns_uri is None else root.find(f".//{q('HEADER')}")
    if header is not None:
        prop_last = None
        for prop in header.findall(q("PROPERTY")):
            if prop.get("NAME") == "lastUsedAnnotationId":
                prop_last = prop
                break
        if prop_last is None:
            prop_last = ET.SubElement(header, q("PROPERTY"), {"NAME": "lastUsedAnnotationId"})
        prop_last.text = str(max_id_num)

    # ---------- Save ----------
    tree.write(outputfilename, encoding="utf-8", xml_declaration=True)
    print(f"\n✅ Language annotation saved to: {outputfilename}")

# Main function

In [ ]:
def process_files(file_names, clitics_dict, word_morph_gloss_dict, lexicon_dict):
    """
    file_names: list of base filenames, e.g. ["pilar-i014", "pilar-i025"]

    Uses pilar_files (dict) to get speakers.
    Steps 1–5 write to /tmp and are “hidden”.
    Step 6 writes the final file: [base_name]_automatic.eaf in the current directory.

    Returns: dict { base_name: final_output_filename }
    """

    results = {}

    for base_name in file_names:
        speakers = pilar_files[base_name]

        # Original input file in the working directory
        file_new = base_name + ".eaf"

        # Temporary stem for intermediate files
        tmp_stem = f"/tmp/{base_name}"
        suffix = ".eaf"

        # Steps 1–5 go to /tmp
        file_step1, file_step2, file_step3, file_step4, file_step5 = [
            f"{tmp_stem}_step{i}{suffix}" for i in range(1, 6)
        ]

        # Final output file in working directory
        file_step6 = f"{base_name}_automatic.eaf"

        config = {
            "triplets": [
                (
                    f"{sp}-words",
                    f"{sp}-morph",
                    f"{sp}-gloss",
                    f"{sp}-language"
                )
                for sp in speakers
            ]
        }

        # === Processing chain ===
        clitics(file_new,  file_step1, config)
        clitics_reverse(file_step1, file_step2, config, clitics_dict)
        populate_missing_morph_gloss_from_tokens(
            file_step2, file_step3, config, word_morph_gloss_dict
        )
        fix_person_number_glosses_by_token(
            file_step3, file_step4, config
        )
        fill_empty_glosses_from_lexicon(
            file_step4, file_step5, config, lexicon_dict
        )
        # Step 6: write only the final, user-visible file
        annotate_language_from_morphs_config(
            file_step5, file_step6, config
        )

        results[base_name] = file_step6

    return results

# Apply annotation

In [ ]:
selected = ["pilar-i016", "pilar-i017"]

outputs = process_files(
    selected,
    clitics_dict,
    word_morph_gloss_dict,
    lexicon_dict
)

# Step-by-step annotation of a single file

In [ ]:
file_new = "pilar-i025"

In [ ]:
speakers = pilar_files[file_new]
#speakers = ["A", "B"]

In [ ]:
file_step1, file_step2, file_step3, file_step4, file_step5, file_step6 = [
    f"{file_new}_step{i}{".eaf"}" for i in range(1, 7)
]

In [ ]:
file_new = file_new + ".eaf"

In [ ]:
config = {
    "triplets": [
        (
            f"{sp}-words",
            f"{sp}-morph",
            f"{sp}-gloss",
            f"{sp}-language"
        )
        for sp in speakers
    ]
}

In [ ]:
#Only if tier names are irregular:

#config = {
#    "triplets": [
#        ("Speaker3-words", "Speaker3-morph", "Speaker3-gloss", "Speaker3-language"),
#        ("Speaker2-words",    "Speaker2-morph",    "Speaker2-gloss",    "Speaker2-language"),
#    ],
#}

In [ ]:
#clitics(file_new, file_step1, config)

In [ ]:
#clitics_reverse(file_step1, file_step2, config, clitics_dict)

In [ ]:
#populate_missing_morph_gloss_from_tokens(file_step2, file_step3, config, word_morph_gloss_dict)

In [ ]:
#fix_person_number_glosses_by_token(file_step3, file_step4, config)

In [ ]:
#fill_empty_glosses_from_lexicon(file_step4, file_step5, config, lexicon_dict)


In [ ]:
#annotate_language_from_morphs_config(file_step5, file_step6, config)